# SQL-LoRA: Fine-Tune Llama-3.2-3B for Text-to-SQL

Fine-tunes a 4-bit quantized LLM with QLoRA on the `b-mc2/sql-create-context` dataset. Designed for Google Colab free tier (T4 GPU).

## Setup
Run this notebook in Colab: `Runtime → Change runtime type → T4 GPU`.

In [ ]:
# @title Install dependencies
!pip install unsloth
!pip install --upgrade --quiet datasets huggingface_hub

In [ ]:
# @title Imports & config
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from huggingface_hub import notebook_login

BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
MAX_SEQ_LENGTH = 512
DATASET_ID = "b-mc2/sql-create-context"

In [ ]:
# @title Load base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

In [ ]:
# @title Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_rslora=False,
    use_gradient_checkpointing="unsloth",
)

In [ ]:
# @title Load & format dataset
dataset = load_dataset(DATASET_ID, split="train")

def format_prompt(example):
    return {
        "text": tokenizer.apply_chat_template([
            {"role": "user", "content": f"{example['context']}\n\nQuestion: {example['question']}"},
            {"role": "assistant", "content": example['answer']},
        ], tokenize=False)
    }

dataset = dataset.map(format_prompt)

In [ ]:
# @title Train
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        max_steps=200,
        logging_steps=10,
        save_steps=50,
        output_dir="checkpoints",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        report_to="none",
    ),
    max_seq_length=MAX_SEQ_LENGTH,
)
trainer.train()

In [ ]:
# @title Save adapter & push to Hub
model.save_pretrained_merged("sql_lora_adapter", tokenizer, save_method="lora")
model.push_to_hub_merged("YOUR_USERNAME/sql-lora-llama3.2-3b", tokenizer, save_method="lora")

In [ ]:
# @title Quick eval on a few examples
FastLanguageModel.for_inference(model)
test_prompt = ""
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0]))